# Colab GPU Training for RESD

Notebook flow:
1. mount Google Drive for checkpoints,
2. install missing dependencies,
3. optionally set `HF_TOKEN`,
4. clone the repo from GitHub,
5. warm up the RESD dataset cache from Hugging Face,
6. run training via CLI with Hydra overrides on GPU.

Before running:
- in Colab select `Runtime -> Change runtime type -> GPU`,
- update `REPO_URL` below if you train from your own fork,
- if you want persistent checkpoints, keep the Drive mount cell enabled.


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "hydra": "hydra-core",
    "omegaconf": "omegaconf",
    "datasets": "datasets",
    "huggingface_hub": "huggingface_hub",
    "sklearn": "scikit-learn",
    "torchaudio": "torchaudio",
    "soundfile": "soundfile",
}

missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Installed:" if missing else "Already available:", missing)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/aibryanov/speech-emo-recognition.git"
REPO_REF = "master"
HF_DATASET_NAME = "Aniemore/resd"

REPO_DIR = Path("/content/speech-emo-recognition")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/ser_checkpoints")

EXPERIMENT_NAME = "resd_colab_baseline"
EPOCHS = 5
BATCH_SIZE = 32
LOG_EVERY = 50

CLI_OVERRIDES = [
    "dataset=resd",
    "train.device=cuda",
    "model.num_classes=7",
    "dataset.sample_rate=16000",
    f"experiment_name={EXPERIMENT_NAME}",
    f"train.epochs={EPOCHS}",
    f"dataloader.batch_size={BATCH_SIZE}",
    f"train.log_every={LOG_EVERY}",
    f"train.checkpoint.dir={CHECKPOINT_DIR.as_posix()}",
]

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

print("Repo:", REPO_URL)
print("Checkpoints:", CHECKPOINT_DIR)
print("Overrides:")
print(*CLI_OVERRIDES, sep="\n")


In [ ]:
from getpass import getpass
import os

if not os.getenv("HF_TOKEN"):
    token = getpass("HF token (optional, press Enter to skip): ")
    if token:
        os.environ["HF_TOKEN"] = token

token = os.getenv("HF_TOKEN")
print("HF_TOKEN set:" , bool(token))
if token:
    print(token[:6] + "..." + token[-4:])


In [ ]:
run(["nvidia-smi"])


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)])
print(f"Repo cloned to: {REPO_DIR}")


In [ ]:
from datasets import load_dataset
import os

dataset = load_dataset(HF_DATASET_NAME, token=os.getenv("HF_TOKEN") or None)
print(dataset)


In [ ]:
cmd = [sys.executable, "main.py", *CLI_OVERRIDES]
run(cmd, cwd=REPO_DIR)


In [ ]:
outputs_dir = REPO_DIR / "outputs"

if outputs_dir.exists():
    print("Hydra outputs:")
    for path in sorted(outputs_dir.rglob("*"))[-20:]:
        print(path)
else:
    print("Hydra outputs directory was not found.")

print("\nLatest checkpoints:")
for path in sorted(CHECKPOINT_DIR.rglob("*"))[-20:]:
    print(path)
